<img src="http://wandb.me/logo-im-png" width="400" alt="Weights & Biases" />

<!--- @wandbcode{artifacts-fundamentals} -->


# W&B ワークショップ: 水生生物の分類

### 最初のベースラインから、完全な MLOps を備えたプロダクションレディなモデルまで

---

### 前提シナリオ

**シナリオ:** あなたは海洋生物学 AI 研究チームの一員として、水中写真から水生生物を識別する画像分類器を構築しています。目的は、海洋研究者が海洋生物の多様性を自動的にカタログ化・モニタリングできるよう支援することです。

**あなたの旅:**
1. 実験追跡と視覚的な診断機能を備えた **ベースラインモデルの学習**
2. 学習データへのリネージを持つバージョン管理された Artifact として **モデルをパッケージ化**
3. Model Registry にベースラインを **ステージング** し、より良いハイパーパラメータを探すために **Sweep** を実行
4. Sweep の結果に基づいて、勝者をプロダクションに **昇格**

---

### このワークショップで学べること

| セクション | トピック | 主要なスキル |
|---------|-------|------------|
| 1 | セットアップ | 環境設定、W&B ログイン、config オブジェクト |
| 2 | データと Artifacts | プリロード済みデータ、use_artifact によるリネージ |
| 3 | データ探索 | EDA テーブル、画像統計、グルーピングとフィルタリング |
| 4 | モデル学習 | Run の構造、PyTorch 学習、混合精度、チェックポイントのロギング、TTL |
| 5 | 視覚的ロギング | 画像、テーブル、ROC 曲線、クラス別メトリクス |
| 6 | Run の再開 | ID で再開、シームレスに学習を継続 |
| 7 | オフラインモード | WANDB_MODE、Run の同期 |
| 8 | モデル Artifact | リネージ付きモデル Artifact、Reference Artifact、use_artifact |
| 9 | Registry | コレクション、リンク、ベースラインのステージング |
| 10 | Sweeps | ハイパーパラメータ最適化、Sweep とベースラインの比較、勝者の昇格 |
| 11 | Sweep の結果 | Sweep とベースラインの比較、勝者をプロダクションへ昇格 |
| 12 | Automations | CI/CD ループ、Registry トリガー、自動化ワークフロー |
| 13 | プログラマティック API（オプション） | Public API クエリ、フィルタ、学習曲線、メタデータ更新 |
| 14 | プログラマティックレポート（オプション） | Reports API、ブロック、PanelGrid、自動化されたドキュメント生成 |
| 15 | SDK 設定リファレンス（オプション） | ネットワーク、git、分散学習、高度な設定 |
| 16 | まとめ | 振り返り、次のステップ |

---

# 1. セットアップ

依存関係のインストール、W&B での認証、実験の設定を行いましょう。

`workshop_utils.py` ファイルが、ML 周りの定型コード（変換、データセットクラスなど）をすべて処理します。

In [ ]:
# 依存関係をまだインストールしていない場合は:
# !pip install -r ../requirements.txt -q

In [ ]:
# インポート
import random
import datetime
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.cuda.amp import GradScaler
import wandb
from datetime import datetime, timedelta
import json
import os

# ワークショップユーティリティ（ML 周りの定型コードを処理）
from workshop_utils import (
    CLASS_NAMES, NUM_CLASSES, DEVICE,
    set_seed, get_transforms,
    create_model, count_parameters,
    train_one_epoch, evaluate,
    generate_run_name,
    AquaticDataset,
    create_dataloaders, create_training_components,
    save_checkpoint, log_checkpoint_artifact,
    create_prediction_images, create_predictions_table,
    compute_and_log_class_metrics, prepare_model_files,
    create_sweep_components,
    promote_sweep_winner, promote_baseline,
)

print(f"PyTorch: {torch.__version__}")
print(f"Device: {DEVICE}")
print(f"W&B: {wandb.__version__}")


## 🪄 `.env` の設定と W&B へのログイン

次のセルを実行する前に、このディレクトリの `.env` ファイルを開いて、ご自身の値を入力してください:
- **`YOUR_NAME`** - 名前（小文字、スペースなし。例: `alice`）。これによって Artifact や Registry エントリに名前空間が付き、複数の参加者が同じプロジェクトを共有できます。
- **`WANDB_ENTITY`** - W&B チーム名
- **`WANDB_PROJECT`** - プロジェクト名（デフォルト: `SIE-Workshop-2026`）
- **`WANDB_BASE_URL`** - W&B サーバーの URL
- **`WANDB_API_KEY`** - W&B プロフィールの「Settings」セクションで確認できます

In [ ]:
# .env ファイルから環境変数を読み込み
# YOUR_NAME、WANDB_ENTITY、WANDB_PROJECT、WANDB_BASE_URL を読み込みます
# こうしておけば、各ファイルではなく .env だけに 1 回設定しておけば済みます。
from dotenv import load_dotenv
load_dotenv(override=True)

YOUR_NAME = os.environ.get("YOUR_NAME")
WANDB_ENTITY = os.environ.get("WANDB_ENTITY")
WANDB_PROJECT = os.environ.get("WANDB_PROJECT", "SIE-Workshop-2026")
WANDB_HOST = os.environ.get("WANDB_BASE_URL")
WANDB_API_KEY = os.environ.get("WANDB_API_KEY")

if not YOUR_NAME:
    raise ValueError(
        "YOUR_NAME not set in .env. Add YOUR_NAME=alice (lowercase, no spaces). "
        "This namespaces your artifacts and registry entries per participant."
    )

if not WANDB_ENTITY:
    raise ValueError("WANDB_ENTITY not set in .env. Add your W&B team name.")

# W&B で認証
wandb.login(host=WANDB_HOST, key=WANDB_API_KEY)

print(f"Name:    {YOUR_NAME}")
print(f"Entity:  {WANDB_ENTITY}")
print(f"Project: {WANDB_PROJECT}")
print(f"Host:    {WANDB_HOST}")

## 設定

すべてのハイパーパラメータと Run のメタデータを単一の config オブジェクトにまとめます。ランダムなオープンソースモデルが割り当てられます。

In [ ]:
# モデルをランダムに割り当て - W&B での比較用に多様な Run を作成します！
WORKSHOP_MODELS = ["resnet50", "efficientnet_b0"]
ASSIGNED_MODEL = random.choice(WORKSHOP_MODELS)
print(f"Your assigned model: {ASSIGNED_MODEL}")

# 学習設定 - これらは自動的に W&B にロギングされます
CONFIG = {
    "user_name": YOUR_NAME,    # 参加者ごとに Artifact と Registry に名前空間を付与
    "model_name": ASSIGNED_MODEL,
    "num_classes": NUM_CLASSES,
    "epochs": 3,              # ワークショップ用に短時間学習
    "batch_size": 32,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "image_size": 224,
    "max_samples": 1000,      # 高速にイテレーションするためのサブセット
    "use_amp": True,
    "seed": 42,
}

# 再現性のために乱数シードを設定
set_seed(CONFIG["seed"])
print(f"\nConfig: {CONFIG}")
print(f"Using device: {DEVICE}")

---

# 2. データ準備

AQUA データセットは **事前に準備され、W&B の Artifact としてロギングされ、ローカル環境にプリロード** されています。これは、共有ストレージ（NFS、S3、チームドライブ）にデータが配置され、再ダウンロードすることなく W&B でデータを **追跡・バージョン管理** する、本番でよくあるパターンを再現しています。

**ローカルの `data/` ディレクトリの中身:**
- `data/train/` - 学習スプリット（約 6,500 枚、20 クラスのサブフォルダ）
- `data/val/` - 検証スプリット（約 800 枚）
- `data/test/` - テストスプリット（約 800 枚）

**W&B 内のデータ（同じデータが Artifact としてバージョン管理されています）:**
- `aqua-train:v0`, `aqua-val:v0`, `aqua-test:v0`

**重要な W&B のコンセプト:** 学習 Run の中で `use_artifact()` を呼び出して、Run が特定のデータセットバージョンに **依存している** ことを宣言します。これにより、どのデータがどのモデルを学習したかを示すグラフ、つまり W&B 上の **リネージ** が作成されます。実際のデータはローカルディスクから読み込まれ、`use_artifact()` は追跡の役割を担います。

In [ ]:
# W&B Artifact のパス（リネージ追跡用）
ARTIFACT_PROJECT = f"{WANDB_ENTITY}/{WANDB_PROJECT}"
TRAIN_ARTIFACT = f"{ARTIFACT_PROJECT}/aqua-train:latest"
VAL_ARTIFACT   = f"{ARTIFACT_PROJECT}/aqua-val:latest"
TEST_ARTIFACT  = f"{ARTIFACT_PROJECT}/aqua-test:latest"
WEIGHTS_ARTIFACT = f"{ARTIFACT_PROJECT}/pretrained-{CONFIG['model_name']}:latest"

# ローカルデータのパス（ワークショップ環境にプリロード済み）
DATA_ROOT = "./data"
LOCAL_TRAIN_DIR   = f"{DATA_ROOT}/train"
LOCAL_VAL_DIR     = f"{DATA_ROOT}/val"
LOCAL_TEST_DIR    = f"{DATA_ROOT}/test"
LOCAL_WEIGHTS_DIR = "./pretrained_weights"

# ローカルデータが存在するか確認
for name, path in [("Train", LOCAL_TRAIN_DIR), ("Val", LOCAL_VAL_DIR),
                   ("Test", LOCAL_TEST_DIR), ("Weights", LOCAL_WEIGHTS_DIR)]:
    status = "OK" if os.path.exists(path) else "MISSING"
    print(f"  {name}: {path} [{status}]")

In [ ]:
# 水中画像向けに最適化されたデータ拡張を含みます: 色のジッター、回転など
print(f"Image transforms ready (size: {CONFIG['image_size']}x{CONFIG['image_size']})")

# ダウンロード済み Artifact 内のクラスフォルダから画像を読み込みます
print("AquaticDataset class ready")

In [ ]:
# セクション 2 完了 - 変換と Dataset クラスの準備が整いました
#
# 実際のデータ読み込みは学習 RUN（セクション 4）で行います:
# 1. wandb.init() で学習 Run を開始
# 2. use_artifact() でデータセット依存関係を宣言（リネージを作成！）
# 3. ローカルディスクからデータを読み込み（環境にプリロード済み）
# 4. メトリクスを W&B にロギングしながら学習を進める
#
# このパターンにより、どのデータセットバージョンが
# 使われたかが学習 Run から正確に分かります - 再現性に不可欠です！

print("Section 2 complete: transforms defined, AquaticDataset ready")
print("Data is pre-loaded locally; use_artifact() will track lineage in W&B")

---

# 3. W&B でのデータ探索

W&B プロジェクト内に EDA テーブルが用意されています！

`<host-url>/<team-name>/SIE-Workshop-2026` に移動してください。"dataset-eda-exploration" の Run を探します

この Run では、以下を含むテーブルがロギングされています:
- 各クラスのサンプル画像
- 画像統計（明度、コントラスト、カラーチャネル）

このテーブルを使って:
1. クラスでグループ化して、種ごとのサンプルを確認します
2. 明度でソートして、暗い画像と明るい画像を見つけます
3. blue_ratio でフィルタリングして、水中特有の青みを検査します
4. 海洋生物種ごとのコントラストを比較します
5. 2D 投影（PCA）を使ってデータセット全体の構造を理解します

---

# 4. `Run` の構造 🩺 とモデル学習


`Run` は、いくつかの特定のデータ構造の中で実験の詳細な記録を保存します。重要なポイントは次のとおりです
- `Run.config` は、入力データへのパスや学習ハイパーパラメータといった Run の設定データを保存する辞書的な構造です。`wandb.init(config=<config-dict>)` に辞書を渡すことで config を初期化できます。
- `Run.history` は、実験中のメトリクスやメディアの履歴値を保存する辞書のリストです。`wandb.log(<metric-dict>)` を呼び出すことで、学習メトリクスの新しいスナップショットを追加できます
- `Run.summary` は、サマリーメトリクスやメディアを記録するための辞書です。デフォルトでは、`summary` には各メトリクスの最新のロギング値が含まれますが、自由に上書きしたり要素を追加したりできます。

**このセクションで取り上げる W&B 機能:**
- `tags` と `group` - フィルタリングや比較のために Run を整理
- `notes` - Run の概要画面に表示される簡単な説明
- `define_metric()` - epoch を X 軸に設定してチャートを見やすく
- `commit=False` - 異なるフェーズのメトリクスを同じステップにロギング
- `use_artifact()` - 自動的にリネージ追跡を行いつつデータ依存関係を宣言
- `wandb.alert()` - 検証精度が改善したら通知を受け取る

In [ ]:
# モデルは次のセル（学習 Run の中）で作成します
# こうすることで、W&B Artifact から事前学習済み重みをリネージ付きで読み込めます。
print(f"Model: {CONFIG['model_name']}")
print(f"Weights artifact: {WEIGHTS_ARTIFACT}")

In [ ]:
# パート 1: W&B Run の初期化
run_name = generate_run_name(CONFIG)

run = wandb.init(
    entity=WANDB_ENTITY,
    project=WANDB_PROJECT,
    name=run_name,
    reinit="create_new",
    job_type="training",
    group=YOUR_NAME, # GROUP: 自分の Run を分離 — UI で YOUR_NAME でフィルタできます
    # TAGS: フィルタ可能なラベル - モデル、データセット、実験タイプで Run を見つける
    tags=[
        YOUR_NAME,                     # 自分の名前 — 自分の Run を探すためのフィルタ
        "AQUA",                        # データセット
        "baseline",                    # 実験タイプ
        CONFIG["model_name"],          # モデルアーキテクチャ
        "workshop-uk-2026",            # ワークショップ識別子
        f"epochs-{CONFIG['epochs']}",  # ハイパーパラメータタグ
    ],
    # NOTES: Run の概要に表示される簡単な説明
    notes=f"Workshop training: {CONFIG['model_name']} on AQUA. "
          f"Epochs: {CONFIG['epochs']}, LR: {CONFIG['learning_rate']}, BS: {CONFIG['batch_size']}",
    config=CONFIG,
    # SHARED MODE: 複数プロセスが同じ Run にロギングできるようにする
    settings=wandb.Settings(
        mode="shared",
        x_label="primary",     # このノードのログ/メトリクスに W&B UI で表示されるラベル
        x_primary=True,        # こちらが primary ノード（config、テレメトリなどをアップロード）
    ),
)

# DEFINE_METRIC: "epoch" を X 軸として設定し、チャートを見やすく
#
# 重要: define_metric は必ず "train/*" や "val/*" のような特定プレフィックスに限定してください。
# run.define_metric("*", step_metric="epoch") は避けてください — 大規模環境では
# Run メタデータが 15MB 制限を超えて肥大化し、フロントエンドのパフォーマンスを著しく低下させます。
run.define_metric("epoch")
run.define_metric("train/loss", step_metric="epoch")
run.define_metric("train/accuracy", step_metric="epoch")
run.define_metric("val/*", step_metric="epoch")
run.define_metric("learning_rate", step_metric="epoch")

# バッチごとのステップメトリクス: train/global_step を X 軸として使用
# (shared モードでは run.log() の `step=` 引数がサポートされないため、
#  ステップをメトリクスとしてロギングし、ここで X 軸として定義します)
#
# hidden=True: これはチャート化する価値のあるメトリクスではなく、ステップカウンタです。
# 非表示にしておくことでワークスペースをクリーンに保てます — 大規模環境で
# 数百のメトリクスを扱う場合、自動生成パネルで UI が散らかるのを防ぎます。
run.define_metric("train/global_step", hidden=True)
run.define_metric("train/loss_step", step_metric="train/global_step")
run.define_metric("train/acc_step", step_metric="train/global_step")

print(f"Run: {run_name}")
print(f"  View at: {run.url}")
print(f"  Tags: {run.tags}")

In [ ]:
# パート 2: Artifact の読み込みとセットアップ
# use_artifact() は LINEAGE を作成 — W&B はこのモデルを学習したデータを正確に追跡します
# データはローカルにプリロード済みなので、use_artifact() はリネージ追跡のためだけに呼び出します

run.use_artifact(TRAIN_ARTIFACT, type='dataset')
run.use_artifact(VAL_ARTIFACT,   type='dataset')
run.use_artifact(TEST_ARTIFACT,  type='dataset')

# プリロード済みのローカルデータを指す
train_dir, val_dir, test_dir = LOCAL_TRAIN_DIR, LOCAL_VAL_DIR, LOCAL_TEST_DIR

# ローカルの事前学習済み重みからモデルを作成（リネージは W&B Artifact で追跡）
model = create_model(
    CONFIG["model_name"], NUM_CLASSES, pretrained=True,
    weights_artifact=WEIGHTS_ARTIFACT, run=run,
    local_weights_dir=LOCAL_WEIGHTS_DIR
)
model = model.to(DEVICE)

train_loader, val_loader, test_loader = create_dataloaders(train_dir, val_dir, test_dir, CONFIG)
criterion, optimizer, scheduler, scaler = create_training_components(model, CONFIG)

# テストデータセットへの参照を保持しておく（セクション 5 の可視化で必要）
test_dataset = AquaticDataset(
    test_dir,
    transform=get_transforms(CONFIG["image_size"], is_training=False),
    class_names=CLASS_NAMES
)

total_params, trainable_params = count_parameters(model)
print(f"\nModel: {CONFIG['model_name']} ({total_params:,} params, {trainable_params:,} trainable)")

In [ ]:
# パート 3: 学習ループ（W&B ロギング）

best_val_acc = 0.0
best_model_path = None

for epoch in range(CONFIG["epochs"]):
    print(f"\nEpoch {epoch+1}/{CONFIG['epochs']}")

    # SPARSE LOGGING: log_interval でバッチ単位のメトリクスを送る頻度を制御します。
    # 大規模環境では毎バッチのロギングはコストが高い — 代わりに N ステップごとにロギングします。
    # ここでは短いベースラインのため log_interval=1、Sweep では log_interval=5 を使います。
    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, scaler, DEVICE,
        epoch, log_interval=1, run=run
    )
    val_loss, val_acc, val_preds, val_labels, val_probs = evaluate(
        model, val_loader, criterion, DEVICE, desc=f"Epoch {epoch+1} [Val]"
    )
    scheduler.step()

    # ── STEP CONTROL: commit=False を使うと、各エポックのメトリクスを同じステップに保てます ──
    # これなしだと、run.log() を呼ぶたびに新しいステップが作られて → チャートが不揃いになります！
    #
    # このパターンは大規模環境での混合周波数ロギングでも重要です:
    # 安価なスカラーは毎ステップ、高価なメディア（画像、テーブル）はもっと低頻度でロギング。
    # commit=False でまとめて、各ステップの最後の呼び出しだけ commit=True にします。
    run.log({"epoch": epoch + 1, "train/loss": train_loss, "train/accuracy": train_acc}, commit=False)
    run.log({"val/loss": val_loss, "val/accuracy": val_acc}, commit=False)
    run.log({"learning_rate": scheduler.get_last_lr()[0]})  # commit=True → ステップが進みます

    print(f"  Train: {train_loss:.4f} loss, {train_acc:.2f}% acc")
    print(f"  Val:   {val_loss:.4f} loss, {val_acc:.2f}% acc")

    # ── ベストモデルの追跡 ──────────────────────────────────────────────────
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_path = f"best_model_epoch{epoch+1}.pth"
        save_checkpoint(model, optimizer, CONFIG, epoch+1, val_acc, val_loss, best_model_path)

        # ALERT: 検証精度が改善したときに通知
        run.alert(
            title="New Best Model!",
            text=f"Validation accuracy improved to {val_acc:.2f}% at epoch {epoch+1}",
            level=wandb.AlertLevel.INFO
        )
        run.summary.update({"best_val_accuracy": val_acc, "best_val_loss": val_loss, "best_epoch": epoch + 1})

    # ── チェックポイント Artifact のロギング（バージョン管理 + TTL 付き） ────────────────────────
    log_checkpoint_artifact(
        run, model, optimizer, CONFIG, epoch+1,
        metrics={"val_accuracy": val_acc, "val_loss": val_loss,
                 "train_accuracy": train_acc, "train_loss": train_loss},
        is_best=(val_acc >= best_val_acc),
        is_last=(epoch == CONFIG["epochs"] - 1),
    )

print(f"\nTraining complete! Best val accuracy: {best_val_acc:.2f}%")

In [ ]:
# TTL の実例: API 経由で Artifact の TTL を確認・変更する
# 学習中に TTL=7 日を設定しましたが、もし "best" チェックポイントが
# 重要であることが分かったらどうしますか？ Public API を使って後から TTL を
# 延長したり解除したりできます。

ttl_days = 60

api = wandb.Api()
best_checkpoint_path = f"{WANDB_ENTITY}/{WANDB_PROJECT}/model-{YOUR_NAME}-{CONFIG['model_name']}:best"

try:
    best_checkpoint = api.artifact(best_checkpoint_path)
    print(f"Best checkpoint: {best_checkpoint.name}:{best_checkpoint.version}")
    print(f"  Current TTL: {best_checkpoint.ttl}")

    best_checkpoint.ttl = timedelta(days=ttl_days)
    best_checkpoint.save()
    print(f"  Updated TTL: {best_checkpoint.ttl}")
except Exception as e:
    print(f"Could not fetch artifact (run training first): {e}")

---

# 5. 視覚的ロギング (メディア)

水中画像、予測結果などのリッチな視覚的診断情報をロギングします。

In [ ]:
# テストセットでの最終評価
print("\nEvaluating on test set...")

checkpoint = torch.load(best_model_path)
model.load_state_dict(checkpoint["model_state_dict"])

test_loss, test_acc, test_preds, test_labels, test_probs = evaluate(
    model, test_loader, criterion, DEVICE, desc="Test Evaluation"
)

print(f"\nTest Results:")
print(f"  Test Loss: {test_loss:.4f}")
print(f"  Test Accuracy: {test_acc:.2f}%")

run.log({
    "test/loss": test_loss,
    "test/accuracy": test_acc
})

run.summary["test_accuracy"] = test_acc
run.summary["test_loss"] = test_loss

In [ ]:
# 画像と信頼度スコア付きで予測サンプルをロギング
# create_prediction_images() が定型処理を行います

prediction_images = create_prediction_images(
    test_dataset, test_preds, test_probs, CLASS_NAMES, n_samples=16
)

# W&B にロギング - 画像は Media タブに表示されます
run.log({"predictions/samples": prediction_images})

print(f"Logged {len(prediction_images)} prediction samples to W&B")


In [ ]:
# 詳細分析のために予測結果を W&B Table としてロギング
# ヒストグラム可視化用にクラス別の信頼度スコアも含めます

predictions_table = create_predictions_table(
    test_dataset, test_preds, test_probs, CLASS_NAMES, n_samples=100
)

# W&B にロギング - テーブルは Tables タブに表示されます
run.log({"predictions/analysis_table": predictions_table})

print(f"Logged predictions table with {len(CLASS_NAMES)} class score columns")
print("  In W&B UI try:")
print("  - Group by 'truth' to see recall per class")
print("  - Group by 'guess' to see precision per class")
print("  - Filter: row['truth'] != row['guess'] to find errors")

In [ ]:
# クラス別メトリクス（precision、recall、F1）を W&B Table としてロギング
# このヘルパーが sklearn メトリクスの計算、テーブル構築、run.summary 更新を行います
f1_macro = compute_and_log_class_metrics(run, test_labels, test_preds, CLASS_NAMES)

## テーブルのロギングモード: MUTABLE と INCREMENTAL

直前にロギングしたテーブルは **IMMUTABLE（変更不可）** です。ロギングされた後は変更できません。これがデフォルトで、Run 終了時のスナップショットには最適です。

ですが、次のようなことをしたい場合はどうでしょうか:
- **後からテーブルを拡張する** — 追加のメトリクスを計算した後で新しい列を追加したい？ それは **MUTABLE** モードです。
- **学習中にテーブルが成長していくのを見る** — バッチごとに行を追加して、結果をリアルタイムにモニタリングしたい？ それは **INCREMENTAL** モードです。

以下では両方のモードを示します。各ステージで一時停止するので、W&B UI を開いてテーブルがライブに変化する様子を確認できます。

In [ ]:
# ── MUTABLE テーブル: 時間をかけて列を追加し、結果を拡張していきます ─────────────────
# ステージ 1: 予測のみをロギング
# ステージ 2: 信頼度スコアを追加
# ステージ 3: 正解/不正解フラグを追加
# W&B UI → 該当 Run → "mutable_evals" テーブルを開いてテーブルが成長する様子を見てください！

# ステージ 1
import time
import numpy as np

mutable_table = wandb.Table(
    columns=["image_idx", "true_class", "predicted_class"],
    log_mode="MUTABLE",
)

# ステージ 1: 予測のみ
for i in range(min(50, len(test_preds))):
    mutable_table.add_data(
        i,
        CLASS_NAMES[test_labels[i]],
        CLASS_NAMES[test_preds[i]],
    )

run.log({"mutable_evaluations": mutable_table})
print("Stage 1: Logged predictions (3 columns)")
print("  → Open W&B UI now. Check the 'mutable_evaluations' table.")

In [ ]:
# ステージ 2: 信頼度スコアを追加
confidences = [float(test_probs[i][test_preds[i]]) for i in range(min(50, len(test_preds)))]
mutable_table.add_column("confidence", [round(c, 4) for c in confidences])
run.log({"mutable_evaluations": mutable_table})
print("Stage 2: Added 'confidence' column (now 4 columns)")
print("  → Refresh the table in W&B — the new column appears.")

In [ ]:
# ステージ 3: 正解/不正解フラグを追加
correct_flags = ["Correct" if test_labels[i] == test_preds[i] else "Wrong"
                 for i in range(min(50, len(test_preds)))]
mutable_table.add_column("result", correct_flags)
run.log({"mutable_evaluations": mutable_table})
print("Stage 3: Added 'result' column (now 5 columns)")
print("  → Refresh again — filter by 'result' = 'Wrong' to see misclassifications.")
print("\nMUTABLE example complete. Table was updated in-place 3 times.")

In [ ]:
# ── INCREMENTAL テーブル: 行がバッチ単位で表示されるのを観察 ───────────────────────────
# 長時間ジョブで予測がバッチ単位で到着するケースをシミュレートします。
# W&B UI → 該当 Run → "incremental_predictions" テーブルを開いてください。
# UI のステップスライダーで増分をスクラブできます！

incr_table = wandb.Table(
    columns=["batch", "image_idx", "true_class", "predicted_class", "confidence"],
    log_mode="INCREMENTAL",
)

BATCH_SIZE = 5
num_samples = min(20, len(test_preds))
num_batches = num_samples // BATCH_SIZE

print(f"Logging {num_samples} predictions in {num_batches} batches of {BATCH_SIZE}")
print("  → Open W&B UI now. Watch the 'incremental_predictions' table grow.\n")

for batch_idx in range(num_batches):
    start = batch_idx * BATCH_SIZE
    end = start + BATCH_SIZE

    for i in range(start, end):
        incr_table.add_data(
            batch_idx + 1,
            i,
            CLASS_NAMES[test_labels[i]],
            CLASS_NAMES[test_preds[i]],
            round(float(test_probs[i][test_preds[i]]), 4),
        )

    run.log({"incremental_predictions": incr_table})
    print(f"  Batch {batch_idx + 1}/{num_batches}: {end} rows total")
    time.sleep(10)  # 各バッチが UI に届く様子を見られるように一時停止

print(f"\nINCREMENTAL demo complete. {num_samples} rows logged across {num_batches} batches.")
print("  → In the W&B UI, use the step slider below the table to scrub through batches.")

## ここで一旦停止 — Shared モードのデモ

**次のセルはまだ実行しないでください。**

学習 Run はまだアクティブな状態です。今度は **新しいターミナル** を開いて、（スクリプト内の Run ID を更新した上で）次を実行してください:

```bash
python shared_worker.py
```

これにより、**同じ Run** にロギングする shared モードのワーカーが起動します。W&B UI を確認すると、2 つのプロセスからメトリクスが届いているのが分かります。ワーカー側のメトリクスは `worker` セクションの下にロギングされます

ワーカーが完了したら戻ってきて、残りのセルを実行してください。

In [ ]:
# W&B の組み込みインタラクティブチャートで ROC 曲線をロギング
# 各種について one-vs-rest の ROC 曲線を作成し、
# モデルが各種を他のすべてからどれだけうまく識別できているかを示します。
# W&B UI 上で完全にインタラクティブです: ホバー、クラスの切り替え、自動 AUC 計算が可能です。

run.log({
    "evaluation/roc_curve": wandb.plot.roc_curve(
        y_true=test_labels,
        y_probas=test_probs,
        labels=CLASS_NAMES,
        title="AQUA Species ROC Curves"
    )
})

# 完了する前に Run ID を保存 — 後で再開する際に必要です
training_run_id = run.id
print(f"Run ID saved for resume: {training_run_id}")

run.finish()

**ROC 曲線の色を修正する**

ROC チャートのすべての線が同じ色（ピンク）になっているのに気付きましたか？ これは、W&B が *クラス* ではなく *Run* ごとに色付けするためで、20 種すべてが単一の Run から来ているので、すべて同じ Run の色を継承してしまっています。

Vega ベースのカスタムチャートは完全にカスタマイズ・再利用可能で、各種に独自の色を割り当てられます:
1. **AQUA Species ROC Curves** チャートにマウスをホバーし、**ギア ⚙ アイコン** をクリック
2. **Vega spec** タブの横の Edit を選択
3. 91〜99 行目を探します


```json
      "encoding": {
        "color": {
          "type": "nominal",
          "field": "name",
          "scale": {
            "range": {
              "field": "color"
            }
          },
```

を

```json
      "encoding": {
        "color": {
          "type": "nominal",
          "field": "class",
          "scale": {"range": "category"},
```

に変更します。

---

# 6. Run の再開

今、3 エポックの学習が終わって `run.finish()` を呼びました。しかし、チャートを見て「もう少し学習が必要だ」と判断したらどうしますか？ あるいは、学習スクリプトがエポック 2 でクラッシュして、続きから再開したい場合は？

W&B では Run の ID で **Run を再開** できます。再開された Run は同じ履歴にロギングを続けるので、チャートにはエポック 1 からエポック 10 まで途切れのない 1 本の線が表示されます（2 本の別々の Run にはなりません）。

鍵になるのは `resume="must"` です:
- `"must"` — Run が **必ず** 既に存在している必要があります（そうでなければエラー）
- `"allow"` — Run があれば再開、なければ新しい Run を作成
- `"auto"` — 同じファイルシステムから自動的に再開

モデル、オプティマイザ、データはまだメモリに残っているので（カーネルは再起動されていません）、Run に再接続して学習を続けるだけで十分です。実際のクラッシュリカバリのシナリオでは、保存されたチェックポイントから再読み込みすることになります。パターンは同じで、`model.load_state_dict()` を追加するだけです。

In [ ]:
# 学習 Run を再開して 10 エポックまで続ける
TOTAL_EPOCHS = 10

run = wandb.init(
    entity=WANDB_ENTITY,
    project=WANDB_PROJECT,
    id=training_run_id,   # 先ほどと同じ Run ID
    resume="must",        # 必ず存在している必要があります — なければエラー
)

# define_metric を再宣言 — これらはクライアント側の指示で、
# セッションをまたいで保持されません。実際の学習スクリプトでは
# 既にコード内に書かれているはずなので、再開は「そのまま動く」のが普通です。
run.define_metric("epoch")
run.define_metric("train/loss", step_metric="epoch")
run.define_metric("train/accuracy", step_metric="epoch")
run.define_metric("val/*", step_metric="epoch")
run.define_metric("learning_rate", step_metric="epoch")
run.define_metric("train/global_step")
run.define_metric("train/loss_step", step_metric="train/global_step")
run.define_metric("train/acc_step", step_metric="train/global_step")

print(f"Resumed run: {run.id}")
print(f"Training epochs {CONFIG['epochs']+1} → {TOTAL_EPOCHS}\n")

for epoch in range(CONFIG["epochs"], TOTAL_EPOCHS):
    print(f"\nEpoch {epoch+1}/{TOTAL_EPOCHS}")

    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, scaler, DEVICE,
        epoch, log_interval=1, run=run
    )
    val_loss, val_acc, val_preds, val_labels, val_probs = evaluate(
        model, val_loader, criterion, DEVICE, desc=f"Epoch {epoch+1} [Val]"
    )
    scheduler.step()

    run.log({"epoch": epoch + 1, "train/loss": train_loss, "train/accuracy": train_acc}, commit=False)
    run.log({"val/loss": val_loss, "val/accuracy": val_acc}, commit=False)
    run.log({"learning_rate": scheduler.get_last_lr()[0]})

    print(f"  Train: {train_loss:.4f} loss, {train_acc:.2f}% acc")
    print(f"  Val:   {val_loss:.4f} loss, {val_acc:.2f}% acc")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_path = f"best_model_epoch{epoch+1}.pth"
        save_checkpoint(model, optimizer, CONFIG, epoch+1, val_acc, val_loss, best_model_path)
        run.summary.update({"best_val_accuracy": val_acc, "best_val_loss": val_loss, "best_epoch": epoch + 1})

print(f"\nResumed training complete! Best val accuracy: {best_val_acc:.2f}%")
run.finish()

---

# 7. オフラインモード

すべての学習環境にインターネットアクセスがあるわけではありません。ファイアウォール越しの GPU クラスタで作業していたり、コンピュートノードにはアクセスできるが W&B インスタンスにはアクセスできなかったりするかもしれません。W&B は **オフラインモード** でこれに対応しています。すべてのメトリクス、Artifact、システム統計情報がローカルディレクトリに保存されます。オンラインに戻ったら、1 つのコマンドですべてを W&B に同期できます。

Run を初期化する際に `mode="offline"` を設定します。W&B はサーバーに送信する代わりに、すべてをローカルに書き込みます。以下では、オフラインで短い学習ループを実行した後、結果を同期します。

In [ ]:
# オフラインモード — すべてローカルに保存され、後で同期されます
offline_run = wandb.init(
    entity=WANDB_ENTITY, project=WANDB_PROJECT,
    name=f"offline-training-run-{YOUR_NAME}",
    mode="offline",               # <-- セクション 4 との唯一の違い
    group=YOUR_NAME,
    config=CONFIG,
    tags=[YOUR_NAME, "AQUA", "offline-demo", CONFIG["model_name"]],
)

# 学習はまったく同じように動きます — データはサーバーではなくディスクへ
for epoch in range(2):
    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, scaler, DEVICE,
        epoch, log_interval=5, run=offline_run
    )
    val_loss, val_acc, _, _, _ = evaluate(
        model, val_loader, criterion, DEVICE, desc=f"Epoch {epoch+1} [Val]"
    )
    offline_run.log({
        "epoch": epoch + 1,
        "train/loss": train_loss, "train/accuracy": train_acc,
        "val/loss": val_loss, "val/accuracy": val_acc,
    })

offline_run.finish()
print("Run saved locally. Sync with: wandb sync <path-printed-above>")

### オフライン Run の同期

オフライン Run を W&B に同期するには、ターミナルで次のコマンドを実行します:

wandb sync `上で表示されたオフライン Run のパス`


---

# 8. リネージ付きモデル Artifact

セクション 2 で既に **データの Artifact** をロギングしました:
- 元データセットの Artifact (`aqua-raw`)
- リネージ付きの分割 Artifact (`aqua-train`, `aqua-val`, `aqua-test`)

いよいよ、学習データを参照する **学習済みモデルを Artifact としてロギング** して、リネージを完成させましょう！

### 完全なリネージチェーン

```
元データセット → 学習/検証/テストの分割 → モデル
```

モデルをロギングする際に `use_artifact()` で分割 Artifact を消費することで、このモデルがどのデータで学習されたのかを正確に示す完全な監査証跡が作成されます。

In [ ]:
# モデル ARTIFACT - リネージチェーンを完成させる
# Run を開始し、データ依存関係（リネージ）を宣言してから、学習済みモデルをロギング。

model_artifact_run = wandb.init(
    entity=WANDB_ENTITY,
    project=WANDB_PROJECT,
    name=f"aqua-model-artifact-logging-{YOUR_NAME}",
    job_type="model-logging",
    group=YOUR_NAME,
    tags=[YOUR_NAME, "aqua", "model-artifact", "baseline"],
    notes=f"Log trained model artifact with lineage to training data splits ({YOUR_NAME})"
)

# use_artifact() で LINEAGE を作成 — W&B が Splits → Model を追跡します
train_artifact_ref = model_artifact_run.use_artifact(TRAIN_ARTIFACT, type='dataset')
val_artifact_ref = model_artifact_run.use_artifact(VAL_ARTIFACT, type='dataset')

# モデルファイルを準備（チェックポイントをコピー + model_info.json を書き出し）
artifact_metadata = prepare_model_files(
    CONFIG, best_model_path, best_val_acc, test_acc, test_loss, f1_macro,
    total_params, trainable_params, train_artifact_ref, val_artifact_ref,
    model_artifact_run.id,
)

# モデル Artifact を作成してロギング（参加者ごとに名前空間を分離）
model_artifact = wandb.Artifact(
    name=f"aqua-species-classifier-{YOUR_NAME}",
    type="model",
    description=f"Aquatic Species classifier ({CONFIG['model_name']}) trained on AQUA dataset",
    metadata=artifact_metadata,
)

# Reference Artifact: アップロードせずに、パス + チェックサムでファイルを追跡
# (本番では add_file() で W&B ストレージにアップロードしますが、
#  ワークショップではネットワーク負荷を抑えるために参照を使います)
model_artifact.add_reference(f"file://{os.path.abspath('artifacts/model.pth')}")
model_artifact.add_reference(f"file://{os.path.abspath('artifacts/model_info.json')}")

model_artifact_run.log_artifact(model_artifact, aliases=["latest", "candidate", "baseline"], tags=["baseline-model", "aqua", "marine-biology"])

print(f"\nModel artifact logged: aqua-species-classifier-{YOUR_NAME} (reference, no upload)")
print(f"  Test Accuracy: {test_acc:.2f}%, Aliases: latest, candidate, baseline")

wandb.finish()

In [ ]:
# Artifact リネージのまとめ
# この時点で、以下の Artifact リネージを構築しました:

print("Artifact lineage chain:")
print("  dataset-upload (run)")
print("    -> aqua-raw-dataset:v0")
print("      -> dataset-splitting (run)")
print("        -> aqua-train:v0, aqua-val:v0")
print(f"          -> aqua-model-artifact-logging-{YOUR_NAME} (run)")
print(f"            -> aqua-species-classifier-{YOUR_NAME}:v0")
print()
print("View lineage: W&B UI > Artifacts > select any artifact > Lineage tab")

## Reference Artifacts: データをコピーせずに外部データを追跡する

これまで作成してきた Artifact はすべて、ファイルを W&B のストレージに **コピー** していました。しかし、データが既に別の場所（S3、GCS、共有ファイルシステムなど）にあり、重複させたくない場合はどうすればよいでしょうか？

**Reference Artifact** がこれを解決します。実際のデータをアップロードせずに、外部ファイル *についての* メタデータとチェックサムをロギングします。W&B は場所、サイズ、整合性を追跡しますが、データそのものは元の場所にとどまります。

サポートされている URI スキーム:
* **file://** -- ローカルファイルシステム
* **http(s)://:** HTTP 経由でアクセス可能なファイルへのパス。HTTP サーバーが ETag および Content-Length レスポンスヘッダーをサポートしている場合、Artifact はチェックサム（etag 形式）とサイズメタデータを追跡します。
* **s3://:** S3 のオブジェクトまたはオブジェクトプレフィックスへのパス。Artifact は参照されるオブジェクトのチェックサムとバージョニング情報（バケットでオブジェクトバージョニングが有効な場合）を追跡します。オブジェクトプレフィックスは、そのプレフィックス配下のオブジェクトを含むように展開されます（デフォルトで最大 100,000 オブジェクト）。
* **gs://:** GCS のオブジェクトまたはオブジェクトプレフィックスへのパス。Artifact は参照されるオブジェクトのチェックサムとバージョニング情報（バケットでオブジェクトバージョニングが有効な場合）を追跡します。オブジェクトプレフィックスは、そのプレフィックス配下のオブジェクトを含むように展開されます（デフォルトで最大 100,000 オブジェクト）。

以下では、既にディスク上にある学習データを使って `file://` でのデモを行います。

In [ ]:
# ============================================================================
# REFERENCE ARTIFACT: 参照によるデータ追跡（コピーなし）
# ============================================================================
# これは、ローカルの学習データを「指し示す」 Artifact を作成しますが、
# W&B にはアップロードしません。W&B はファイルパス、サイズ、
# チェックサムを記録するので、後でデータの整合性を検証できます。
#
# 注意: これはスタンドアロンのデモです - 上記の学習パイプラインや
# リネージチェーンには影響しません。独立した別の Artifact です。

ref_run = wandb.init(
    entity=WANDB_ENTITY,
    project=WANDB_PROJECT,
    name=f"reference-artifact-demo-{YOUR_NAME}",
    job_type="reference-demo",
    group=YOUR_NAME,
    tags=[YOUR_NAME, "aqua", "reference-artifact", "demo"],
    notes=f"Demonstrate reference artifacts pointing to local data ({YOUR_NAME})"
)

# ダウンロード済みの学習データを指す Reference Artifact を作成
# train_dir は先ほど学習 Artifact をダウンロードしたときに設定されました
ref_artifact = wandb.Artifact(
    name=f"aqua-train-reference-{YOUR_NAME}",
    type="reference-dataset",
    description=f"Reference to local training data (no upload, metadata only) ({YOUR_NAME})",
    metadata={
        "source_path": os.path.abspath(train_dir),
        "purpose": "Demonstrates reference artifacts -- data stays on disk",
        "original_artifact": TRAIN_ARTIFACT
    }
)

# add_reference はファイルを場所 + チェックサムで追跡し、アップロードはしません
ref_artifact.add_reference(f"file://{os.path.abspath(train_dir)}")

ref_run.log_artifact(ref_artifact)

print(f"Logged reference artifact: aqua-train-reference-{YOUR_NAME}")
print(f"  Points to: {os.path.abspath(train_dir)}")
print(f"  Data uploaded to W&B: NO (metadata and checksums only)")
print(f"\n  In W&B UI → Artifacts → aqua-train-reference-{YOUR_NAME}:")
print(f"  - Files tab shows referenced paths (not uploaded copies)")
print(f"  - Metadata tab shows source info")

wandb.finish()

---

# 9. Model Registry: ベースラインをステージングする

**Model Registry** はデプロイ用のモデルを管理する場所です。ですが、まだプロダクションに急ぐ必要はありません。ベースラインは手で選んだハイパーパラメータで学習しただけです。まず **ステージング** しておいて、Sweep（セクション 10）でより良い結果が出せるか確かめてから、プロダクションへの昇格を決めましょう。

**重要なコンセプト:**
- **Artifact のエイリアス** = 学習状態（epoch_1、best、latest）
- **Registry のエイリアス** = デプロイ状態（staging、production）

**ワークフロー:**
1. 学習でチェックポイント付きのモデル Artifact を作成
2. 最良の Artifact を **Registered Model**（コレクション）に **staging** としてリンク
3. ハイパーパラメータ Sweep を実行（セクション 10）して、より良いモデルを探す
4. 勝者を **production** に昇格（セクション 11 の終わり）


In [ ]:
# ステップ 1: ベストモデルを REGISTRY にリンクする
# 学習で得たベストチェックポイントを Registered Model にリンクします

registry_run = wandb.init(
    entity=WANDB_ENTITY,
    project=WANDB_PROJECT,
    name=f"registry-promotion-{YOUR_NAME}",
    job_type="registry-promotion",
    group=YOUR_NAME,
    tags=[YOUR_NAME, "registry", "promotion"],
)

# 学習からベストモデル Artifact を取得（参加者ごとに名前空間を分離）
best_model_path = f"{WANDB_ENTITY}/{WANDB_PROJECT}/model-{YOUR_NAME}-{CONFIG['model_name']}:best"
best_artifact = registry_run.use_artifact(best_model_path, type="model")

print(f"Best model artifact: {best_artifact.name}:{best_artifact.version}")
print(f"  Metadata: {best_artifact.metadata}")

# Registry の Registered Model コレクションにリンク
# コレクションが存在しなければ作成されます（参加者ごとに名前空間を分離）
REGISTRY_NAME = f"aqua-classifier-{YOUR_NAME}"  # Registry 内のコレクション名

registry_run.link_artifact(
    artifact=best_artifact,
    target_path=f"wandb-registry-sie-workshop-uk-2026/{REGISTRY_NAME}",
    aliases=["staging"]  # staging から開始
)

print(f"\nLinked to Registry: {REGISTRY_NAME}")
print(f"  Alias: staging")
print(f"  Check Model Registry in W&B UI!")


In [ ]:
# ステップ 2: ステージング済みモデルの確認
# プロダクションに昇格する前に、ステージングしたモデルとベースラインの精度を
# 確認します。セクション 10 で Sweep を実行した後、より良いモデルがあるか
# 確認してから昇格する流れに戻ってきます。

api = wandb.Api()

# モデルを W&B Model Registry から取得（プロジェクト Artifact ストアではなく）。
# UI で Registry -> Models -> Aqua-Classifier を訪問してください。
# "staging" はステップ 1 でリンクするときに割り当てたエイリアスです。
registry_path = f"server/wandb-registry-sie-workshop-uk-2026/{REGISTRY_NAME}:staging"

try:
    staged_artifact = api.artifact(registry_path)

    baseline_val_acc = staged_artifact.metadata.get("val_accuracy", 0)

    print(f"Staged baseline model: {staged_artifact.name}")
    print(f"  Version: {staged_artifact.version}")
    print(f"  Validation accuracy: {baseline_val_acc:.2f}%")
    print(f"  Aliases: {staged_artifact.aliases}")
    print(f"\n  Status: STAGED (not yet promoted to production)")
    print(f"  Next: Run hyperparameter sweep (Section 10) to see if we can beat it")

except wandb.errors.CommError as e:
    print(f"Error: Could not find staged model. Run Step 1 first.")
    print(f"  {e}")

wandb.finish()

print("\nBaseline staged in Registry. Production promotion after sweep (Section 10).")

---

# 10. Sweep によるハイパーパラメータ最適化

ベースラインは Registry に **ステージング** 済みですが、まだプロダクションに昇格していません。昇格する前に、もっと良いハイパーパラメータがないか確認しましょう。学習率を下げてバッチサイズを大きくした方が収束が良いかもしれません。weight decay を強めた方が水中画像にはよいかもしれません。

**W&B Sweeps** はこの探索を自動化します:
1. **探索空間を定義** — どのハイパーパラメータをどの範囲で振るか
2. **戦略を選択** — `random`、`grid`、`bayes`（ベイズ最適化）
3. **agent を起動** — W&B が異なる config で複数の学習ジョブを実行
4. **結果を分析** — すべての Run を W&B UI で並べて比較
5. **勝者を昇格** — 最良のモデル（ベースラインか Sweep）をプロダクションへ

**今回のストーリー:** ステージング済みのベースラインがあります。これから 5 つの実験を異なる学習率、バッチサイズ、weight decay の値で実行します。Sweep の Run がベースラインを上回れば、*そちら* をプロダクションへ昇格します。そうでなければ、ベースラインがプロダクションのバッジを獲得します。

**重要な W&B のコンセプト:**
- `wandb.sweep()` — 探索設定で Sweep controller を作成
- `wandb.agent()` — controller から config を受け取って Run を起動
- Sweep controller が各 Run に異なる `config` 値を自動的に渡します

**実行中の Sweep を制御する:** Sweep agent は呼び出しがブロッキングです。実行中に一時停止・再開・停止したい場合は、**W&B UI**（Sweep ダッシュボードのボタン）または別のターミナルから **CLI** を使います:

* `wandb sweep --pause ENTITY/PROJECT/SWEEP_ID`
* `wandb sweep --resume ENTITY/PROJECT/SWEEP_ID`
* `wandb sweep --stop ENTITY/PROJECT/SWEEP_ID`
* `wandb sweep --cancel ENTITY/PROJECT/SWEEP_ID`

In [ ]:
# ステップ 1: Sweep 設定を定義する

sweep_config = {
    "method": "random",        # ランダム探索 -- 初期探索に効率的
    "name": "aqua-hyperparam-sweep",
    "metric": {
        "name": "val/accuracy",  # 最適化対象
        "goal": "maximize"       # 精度が高いほど良い
    },
    "parameters": {
        # 学習率: 1e-4 から 1e-2 の対数一様分布
        "learning_rate": {
            "distribution": "log_uniform_values",
            "min": 1e-4,
            "max": 1e-2
        },
        # バッチサイズ: 一般的な値をいくつか試す
        "batch_size": {
            "values": [16, 32, 64]
        },
        # weight decay: 1e-5 から 1e-3 の対数一様分布
        "weight_decay": {
            "distribution": "log_uniform_values",
            "min": 1e-5,
            "max": 1e-3
        },
        # 固定パラメータ（Sweep で振らないが、完全性のため含める）
        "model_name": {"value": CONFIG["model_name"]},
        "epochs": {"value": 5},           # ワークショップ用に短く保つ
        "image_size": {"value": 224},
        "max_samples": {"value": 1000},    # ベースラインと同じサブセット
        "use_amp": {"value": True},
    }
}

from pprint import pprint
print("Sweep configuration:")
pprint(sweep_config)

In [ ]:
# ステップ 2: Sweep を作成する
# これにより Sweep が W&B に登録され、sweep_id が返されます。
# Sweep controller は W&B のサーバー上に存在し、agent に config を配ります。

sweep_id = wandb.sweep(sweep_config, project=WANDB_PROJECT, entity=WANDB_ENTITY)

print(f"Sweep created! ID: {sweep_id}")

In [ ]:
# ステップ 3: Sweep 用の学習関数を定義する

# 各 Sweep agent 呼び出しは、この関数を異なる config で実行します。
# ベースライン学習との主な違い:
#   - wandb.init() は明示的な config なしで呼び出されます（Sweep が提供します）
#   - ハイパーパラメータは wandb.config（Sweep controller が設定）から読み取ります
#   - その他はすべて workshop_utils の同じユーティリティを再利用します

def sweep_train(config=None):
    """Sweep agent が呼び出す学習関数。"""
    with wandb.init(
        config=config, group=YOUR_NAME,
        tags=[YOUR_NAME, "aqua", "sweep", CONFIG["model_name"]]
    ) as run:
        # プリエンプション処理: この Sweep Run が中断された場合（スポットインスタンス、
        # SLURM タイムアウトなど）、失敗扱いせずに W&B が自動的に再キューします。
        # Sweep agent は次のイテレーションでこれを拾い直します。
        run.mark_preempting()

        cfg = wandb.config

        # カスタム X 軸を定義（ベースラインと同じ）
        wandb.define_metric("epoch")
        wandb.define_metric("train/*", step_metric="epoch")
        wandb.define_metric("val/*", step_metric="epoch")

        # リネージのために Artifact の使用を宣言（データはローカルにプリロード済み）
        run.use_artifact(TRAIN_ARTIFACT, type="dataset")
        run.use_artifact(VAL_ARTIFACT, type="dataset")

        # セットアップ: データセット、モデル、オプティマイザ（ベースラインと同じ、Sweep からの config）
        model, train_loader, val_loader, criterion, optimizer, scaler = \
            create_sweep_components(cfg, run, WEIGHTS_ARTIFACT, LOCAL_WEIGHTS_DIR)

        # 学習ループ
        best_val_acc = 0.0
        for epoch in range(cfg.epochs):
            train_loss, train_acc = train_one_epoch(
                model, train_loader, criterion, optimizer, scaler, DEVICE,
                epoch, log_interval=5, run=run # log_interval=5 に注目 — スパースロギングです。
            )
            val_loss, val_acc, _, _, _ = evaluate(
                model, val_loader, criterion, DEVICE, desc=f"Epoch {epoch+1}"
            )

            wandb.log({
                "epoch": epoch + 1,
                "train/loss": train_loss,
                "train/accuracy": train_acc,
                "val/loss": val_loss,
                "val/accuracy": val_acc,
            })

            if val_acc > best_val_acc:
                best_val_acc = val_acc

        # ベスト結果を summary にロギング（Sweep が Run のランキングに使用）
        run.summary["best_val_accuracy"] = best_val_acc

print("Sweep training function defined")

In [ ]:
# ステップ 4: Sweep agent を起動する

# これにより、Sweep controller のランダム探索戦略に従って異なるハイパーパラメータで
# 5 つの学習 Run を実行します。
#
# 実行中、W&B の Sweep ダッシュボードで以下を確認できます:
# - 並列座標プロット（どのパラメータの組み合わせが最も効果的か）
# - パラメータ重要度（どのパラメータが最も影響するか）
# - すべての Run を並べて比較

SWEEP_COUNT = 5  # 実行する Run の数

print(f"  Launching {SWEEP_COUNT} sweep runs...")
print(f"  Sweep_path {WANDB_ENTITY}/{WANDB_PROJECT}/{sweep_id}")

wandb.agent(sweep_id, function=sweep_train, count=SWEEP_COUNT)

### オプション - 別の方法: YAML を使った CLI からの Sweep 実行

このノートブックでは、Python で Sweep 設定を定義し、インラインで `wandb.agent()` を呼び出しています。本番では、多くのチームが設定を **YAML ファイル** に定義し、ターミナルから agent を起動します。これにより:

- **並列化** — 複数のターミナルを開いてそれぞれ `wandb agent` を実行すると、すべてが同じ Sweep controller から config を取得します
- **GPU へのピン留め** — 片方のターミナルで `CUDA_VISIBLE_DEVICES=0 wandb agent ...`、もう片方で `CUDA_VISIBLE_DEVICES=1 wandb agent ...`
- **設定とコードの分離** — YAML を別途バージョン管理し、チーム間で共有

`workshop_material/` に 2 つのファイルが用意されています:

- **`sweep_config.yaml`** — Sweep の設定（上記と同じ探索空間を YAML 形式で）
- **`sweep_train.py`** — `wandb.config` から設定を読み込むスタンドアロンの学習スクリプト

試すには（先に `sweep_train.py` の WANDB_ENTITY と WANDB_PROJECT を更新してください）:

```bash
# ターミナル 1: Sweep を作成して agent を開始
cd workshop_material
wandb sweep sweep_config.yaml
wandb agent <ENTITY>/<PROJECT>/<SWEEP_ID>

# ターミナル 2（オプション）: 2 つ目の agent を並列で実行
wandb agent <ENTITY>/<PROJECT>/<SWEEP_ID>
```

両方の agent は、同じ Sweep controller から異なる config を取得します。W&B UI で並列に学習している様子を確認してみてください。

---

# 11. Sweep の結果とベースラインの比較

**次のセルを実行する前に** — Registry 全体に対する automation が事前に設定されています。この Registry のいずれかの Artifact に `production` エイリアスが追加されると、W&B は webhook を発火させ、GitHub Action をトリガーし、レビュー用の Issue が自動的に作成されます。

つまり、下の昇格は、ご自身のコレクション（`aqua-classifier-{YOUR_NAME}`）に対する automation をトリガーすることになります。

- まず、ステージング済みのベースラインを確認してください: Registry > sie-workshop-uk-2026 > `aqua-classifier-{YOUR_NAME}` を訪れて、V0 に `staging` エイリアスが付いていることを確認します
- 下のセルを実行した後、V1 が `production` 付きで表示されることを確認してください
- そして [GitHub Issues タブ](https://github.com/MBakirWB/Automation_Jobs/issues) で automation が動作している様子を確認してください

In [ ]:
# ============================================================================
# Sweep の結果とベースラインの比較 → 勝者をプロダクションに昇格
# ============================================================================
# Sweep が完了したので、ステージング済みベースラインを上回った Sweep Run が
# あったかを確認します。勝者は Registry のプロダクションに昇格されます。

api = wandb.Api()

# Registry からステージング済みベースラインの精度を取得
# まず "staging" を試し、既に昇格されていた場合（再実行時）は "v0" にフォールバック
registry_base = f"wandb-registry-sie-workshop-uk-2026/{REGISTRY_NAME}"
staged_artifact = None
for alias in ["staging", "v0"]:
    try:
        staged_artifact = api.artifact(f"{registry_base}:{alias}")
        break
    except Exception:
        continue

if staged_artifact is None:
    raise RuntimeError(
        f"Could not find baseline in registry '{REGISTRY_NAME}'. "
        "Run Section 9 (Cell 43) first to stage the baseline."
    )

baseline_acc = staged_artifact.metadata.get("val_accuracy", 0)
already_promoted = "production" in staged_artifact.aliases or "staging" not in staged_artifact.aliases

print(f"Baseline: {staged_artifact.name} ({staged_artifact.version})")
print(f"  Validation accuracy: {baseline_acc:.2f}%")
print(f"  Aliases: {staged_artifact.aliases}")
if already_promoted:
    print(f"  (Note: staging alias already removed — this is a re-run)")
print()

# 最良の Sweep Run を見つける
sweep = api.sweep(f"{WANDB_ENTITY}/{WANDB_PROJECT}/{sweep_id}")
sweep_runs = sweep.runs

best_sweep_acc = 0.0
best_sweep_run = None
for run in sweep_runs:
    acc = run.summary.get("best_val_accuracy", 0)
    if acc > best_sweep_acc:
        best_sweep_acc = acc
        best_sweep_run = run

if best_sweep_run:
    print(f"Best sweep run: {best_sweep_run.name}")
    print(f"  Validation accuracy: {best_sweep_acc:.2f}%")
    print(f"  Config: LR={best_sweep_run.config.get('learning_rate'):.5f}, "
          f"BS={best_sweep_run.config.get('batch_size')}, "
          f"WD={best_sweep_run.config.get('weight_decay'):.6f}")

sweep_wins = best_sweep_run and best_sweep_acc > baseline_acc
print(f"\nVerdict: {'SWEEP WINS' if sweep_wins else 'BASELINE HOLDS'}")

### ステップ 2: 勝者をプロダクションに昇格する

上の比較結果に基づいて、勝者を Registry に昇格します:

- **Sweep の勝利:** Sweep 勝者のメタデータを持つ新しい Artifact バージョン (v1) をロギングし、`production` エイリアスで Registry にリンクします。本番では Sweep 中に実際のモデル重みを保存するところですが、ここではバージョニングフローのデモのためにプレースホルダーを使います。
- **ベースラインが勝利:** 既存の v0 を `staging` から `production` に昇格します（新しいバージョンは不要）。

In [ ]:
# このワークショップのカスタム Registry のパス
REGISTRY_PATH = "wandb-registry-sie-workshop-uk-2026"
MIN_ACC_FOR_PRODUCTION = 50.0

if sweep_wins:
    print(f"SWEEP WINS! {best_sweep_acc:.2f}% > {baseline_acc:.2f}%\n")
    promote_sweep_winner(
        YOUR_NAME, CONFIG, WANDB_ENTITY, WANDB_PROJECT,
        REGISTRY_NAME, REGISTRY_PATH, sweep_id,
        best_sweep_run, best_sweep_acc, baseline_acc, staged_artifact,
    )
elif baseline_acc >= MIN_ACC_FOR_PRODUCTION:
    print(f"BASELINE HOLDS! {baseline_acc:.2f}% >= {best_sweep_acc:.2f}%")
    promote_baseline(staged_artifact)
else:
    print(f"Neither model meets the {MIN_ACC_FOR_PRODUCTION}% threshold. No promotion.")

print(f"\nCheck Model Registry in W&B UI → {REGISTRY_NAME}")

---

# 12. Automations: CI/CD ループを閉じる

production エイリアスが追加されたとき、W&B はそのイベントを検知し、事前に設定された webhook を発火させました。その webhook は GitHub リポジトリに repository_dispatch を送信し、GitHub Action をトリガーして、お名前、Artifact バージョン、W&B へのリンクを含むレビュー Issue を作成しました。

ぜひ確認してみてください: [GitHub Issues タブ](https://github.com/MBakirWB/Automation_Jobs/issues)

**完全な CI/CD ループ:**

```
モデル学習 → Artifact をロギング → Registry で昇格 → Automation 発火 → GitHub Issue 作成
```

**Automation をトリガーできるもの:**
- Registry コレクションに新しい Artifact バージョンがリンクされた
- Artifact のエイリアスが追加された（例: `production`、`staging`）
- プロジェクトで新しい Artifact バージョンが作成された

**Automation にできること:**
- イベントの詳細を含む **Slack 通知** を送信
- Artifact のメタデータ、イベントタイプ、作成者を含む JSON ペイロードで **webhook** を呼び出す

W&B UI で automation の履歴を確認できます: プロジェクトまたは Registry → **Automations** タブ → 各 automation をクリックして、実行履歴、ステータス、エラーを確認できます。

独自の automation を設定するには [Automations ドキュメント](https://docs.wandb.ai/models/automations) を参照してください。

---

# 13 - オプション: プログラマティック API: データから意思決定を行う

ベースラインを学習し、Sweep を実行し、モデルを Registry に昇格しました。しかし、本番のワークフローではこれを手動でやることはありません。CI/CD パイプライン、スケジュール済みジョブ、あるいはレビュー用ノートブックから、実験データをプログラマティックにクエリすることになります。

W&B Public API は、ロギングしたすべての内容にフルアクセスできます。これを使って実務で出てくる質問に答えてみましょう。

In [ ]:
api = wandb.Api()

# 1. Sweep の結果を分析用に DataFrame に取り込む
sweep = api.sweep(f"{WANDB_ENTITY}/{WANDB_PROJECT}/{sweep_id}")
rows = []
for run in sweep.runs:
    rows.append({
        "name": run.name,
        "lr": run.config.get("learning_rate"),
        "batch_size": run.config.get("batch_size"),
        "weight_decay": run.config.get("weight_decay"),
        "best_val_acc": run.summary.get("best_val_accuracy", 0),
    })

import pandas as pd
sweep_df = pd.DataFrame(rows).sort_values("best_val_acc", ascending=False)
print(sweep_df.to_string(index=False))

In [ ]:
# 2. MongoDB 風のフィルタで Run をクエリ

# "baseline" タグが付いていて、検証精度が 60% を超える Run をすべて検索
baseline_runs = api.runs(
    f"{WANDB_ENTITY}/{WANDB_PROJECT}",
    filters={
        "$and": [
            {"tags": "baseline"},
            {"summary_metrics.best_val_accuracy": {"$gt": 60}},
        ]
    },
    order="-summary_metrics.best_val_accuracy",  # 上位から
)

print(f"Found {len(baseline_runs)} baseline runs above 60% accuracy\n")
for run in baseline_runs:
    print(f"  {run.name}: {run.summary.get('best_val_accuracy', 0):.2f}% "
          f"(lr={run.config.get('learning_rate')}, bs={run.config.get('batch_size')})")

In [ ]:
# 学習率が 1e-3 未満の resnet50 Sweep Run をすべて検索
precise_runs = api.runs(
    f"{WANDB_ENTITY}/{WANDB_PROJECT}",
    filters={
        "$and": [
            {"tags": "sweep"},
            {"config.model_name": "resnet50"},
            {"config.learning_rate": {"$lt": 1e-3}},
        ]
    },
)

print(f"Found {len(precise_runs)} resnet50 sweep runs with lr < 1e-3")
for run in precise_runs:
    print(f"  {run.name}: lr={run.config['learning_rate']:.6f}, "
          f"acc={run.summary.get('best_val_accuracy', 0):.2f}%")

In [ ]:
# 3. ベースライン vs 最良の Sweep — 完全な学習曲線を比較

# ベースラインの完全な学習履歴（サンプリングなし）を取得
baseline_run = api.run(f"{WANDB_ENTITY}/{WANDB_PROJECT}/{training_run_id}")
baseline_history = baseline_run.scan_history(keys=["epoch", "val/accuracy"])
baseline_points = [(row["epoch"], row["val/accuracy"]) for row in baseline_history if row.get("val/accuracy")]

# 最良の Sweep Run についても同様に
best_sweep_run = sweep.best_run()
sweep_history = best_sweep_run.scan_history(keys=["epoch", "val/accuracy"])
sweep_points = [(row["epoch"], row["val/accuracy"]) for row in sweep_history if row.get("val/accuracy")]

print("Baseline val/accuracy by epoch:")
for epoch, acc in baseline_points:
    print(f"  Epoch {int(epoch)}: {acc:.2f}%")

print(f"\nBest sweep ({best_sweep_run.name}) val/accuracy by epoch:")
for epoch, acc in sweep_points:
    print(f"  Epoch {int(epoch)}: {acc:.2f}%")

In [ ]:
# 4. 事後にメタデータを更新

# Run にプログラマティックにタグを付ける — 評価結果に基づいて
# Run を自動ラベル付けする CI/CD パイプラインで便利
best_sweep_run = sweep.best_run()
best_sweep_run.tags.append("auto-promoted")
best_sweep_run.update()
print(f"Tagged {best_sweep_run.name} as 'auto-promoted'")

---

# 14 - オプション: プログラマティックレポート

学習 Run、Sweep の結果、モデル Artifact がすべてロギングされたので、調査結果をドキュメント化して共有するための **プログラマティックレポート** を作成しましょう。

プログラマティックレポートはコードからレポート作成を **自動化** できます。実験間で一貫性を確保し、リアルタイム更新を可能にし、インタラクティブなダッシュボードをチームと簡単に共有できます。

1 つのレポートをステップごとに構築していきます:
1. レポートを作成し、テキストコンテンツ（**ブロック**）を追加
2. **Panel Grids** を介して、ライブの学習データを **パネル** に取り込み
3. 保存して共有

In [ ]:
import os
import wandb_workspaces.reports.v2 as wr

# Reports API は内部的に wandb.Api() クライアントを作成し、API キーを再検証します。
# WANDB_BASE_URL を設定することで、正しいホストに対して認証されます
# （デフォルトの https://api.wandb.ai ではなく）。
os.environ["WANDB_BASE_URL"] = WANDB_HOST

# ── ステップ 1: レポートを作成 ──────────────────────────────────────────────────
report = wr.Report(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    title=f"AQUA Workshop - Pipeline Report - {YOUR_NAME}",
    description="Programmatic report documenting the aquatic species classification workshop",
)

# ── ステップ 2: コンテンツブロックを追加 ───────────────────────────────────────────────
# 利用可能なブロック: H1、H2、H3、P、UnorderedList、OrderedList、
#   Image、CodeBlock、MarkdownBlock、Link、TableOfContents など
report.blocks = [
    wr.TableOfContents(),
    wr.H1("Aquatic Species Classification"),
    wr.P("This report documents our AQUA workshop pipeline — from data preparation "
         "through model training to hyperparameter optimization and deployment."),
    wr.H1("Workshop Pipeline"),
    wr.P("We followed these steps:"),
    wr.UnorderedList(items=[
        "Consumed pre-prepared dataset artifacts (train/val/test splits)",
        "Trained a baseline model with full experiment tracking",
        "Logged model artifacts with lineage back to training data",
        "Staged the baseline in the Model Registry",
        "Ran hyperparameter sweeps to optimize performance",
        "Promoted the winning model to production",
    ]),
    wr.P(text=["For more details, see the ",
               wr.Link("W&B Reports documentation", url="https://docs.wandb.ai/guides/reports")]),
]

print(f"Report created with {len(report.blocks)} blocks")

Panel Grid でライブデータを引き出す

プログラマティックレポートの真の強みは、W&B プロジェクトから直接データを引き出す **ライブパネル** を埋め込めることです。

- **`PanelGrid`** は `runsets`（どの Run を表示するか）と `panels`（どう可視化するか）を保持します
- **`Runset`** は Run をフィルタリングします。Run 名やタグ（例: `"baseline"`, `"sweep"`）にマッチするように `query` を使います
- **`Panels`** には `LinePlot`、`BarPlot`、`ScatterPlot`、`RunComparer` などがあります

その他のレポート例は [Reports API クイックスタートノートブック](https://colab.research.google.com/github/wandb/examples/blob/master/colabs/intro/Report_API_Quickstart.ipynb) を参照してください

In [ ]:
# ── ステップ 3: ライブ学習データ付きの Panel Grid を追加 ─────────────────────────
# Runset はどの Run を表示するかをフィルタし、panel は可視化方法を選びます。
# レイアウトは 24 列グリッド: w=8 → 1 行に 3 パネル、w=12 → 1 行に 2 パネル
pg = wr.PanelGrid(
    runsets=[
        wr.Runset(WANDB_ENTITY, WANDB_PROJECT, name="Baseline", query=f"{YOUR_NAME}"),
        wr.Runset(WANDB_ENTITY, WANDB_PROJECT, name="Sweep Runs", query="sweep"),
    ],
    panels=[
        # 1 行目: メトリクスチャートを横に 3 つ並べる
        wr.LinePlot(x='epoch', y=['train/loss'], smoothing_factor=0.8,
                    title="Training Loss",      layout={'x': 0,  'y': 0, 'w': 8, 'h': 8}),
        wr.LinePlot(x='epoch', y=['val/loss'], smoothing_factor=0.8,
                    title="Validation Loss",    layout={'x': 8,  'y': 0, 'w': 8, 'h': 8}),
        wr.LinePlot(x='epoch', y=['val/accuracy'],
                    title="Validation Accuracy", layout={'x': 16, 'y': 0, 'w': 8, 'h': 8}),
        # 2 行目: Run Comparer と予測テーブルを横に並べる
        wr.RunComparer(diff_only='split',       layout={'x': 0,  'y': 8, 'w': 12, 'h': 10}),
        wr.WeavePanelSummaryTable(
            table_name="predictions/analysis_table",
                                                layout=wr.Layout(x=12, y=8, w=12, h=10)),
    ]
)

# 既存のブロックに Panel Grid を追加
report.blocks += [
    wr.H1("Training Results — Baseline vs Sweep"),
    wr.P("⭐ Anyone with access can interact with the charts below!"),
    pg,
    wr.H1("Next Steps"),
    wr.P("Share this report with your team using the Share button, or generate a "
         "view-only link for stakeholders who don't have a W&B account."),
]

# ── ステップ 4: レポートを保存 ──────────────────────────────────────────────────
# "readable" はパネルの比率を保ち、"fluid" はブラウザ幅いっぱいに引き伸ばします
report.width = 'fluid'
report.save()

print(f"Report saved with {len(report.blocks)} blocks!")
print(f"View it at: {report.url}")

# 15 - オプション: SDK 設定リファレンス

W&B SDK は `wandb.Settings` を通じて高度にカスタマイズ可能です。これらは、デバッグ、大規模環境でのパフォーマンスチューニング、環境への適応の際に使う「つまみ」です。

設定は `wandb.init(settings=wandb.Settings(...))` に渡すか、`WANDB_` プレフィックスを付けた環境変数で設定できます（例: `WANDB_SILENT=true`）。

完全なリファレンス: [Settings ドキュメント](https://docs.wandb.ai/models/ref/python/experiments/settings)

### リファレンス: 大規模運用・環境向け設定

これらはこの場で意味のあるデモを行うことはできませんが、本番環境では非常に重要です。実際の学習ジョブを動かす際のために [完全な Settings リファレンス](https://docs.wandb.ai/models/ref/python/experiments/settings) をブックマークしておきましょう。

**ネットワークとタイムアウト**

| 設定 | 何をするか | いつ使うか |
|---------|-------------|----------------|
| `x_file_stream_max_line_bytes` | `run.log()` ペイロードサイズの上限（デフォルト 10MB） | 大きなペイロード用に引き上げる。トレードオフ: 大きな呼び出しほど大規模環境で処理が遅くなる |
| `x_graphql_timeout_seconds` | GraphQL リクエストのバックエンドタイムアウト | 大量データでサーバー側タイムアウトが起きるときに引き上げる |
| `x_file_stream_transmit_interval` | SDK がサーバーへデータをフラッシュする頻度 | 小さくする = よりリアルタイム、大きくする = ネットワーク負荷の低減 |
| `http_proxy` / `https_proxy` | W&B トラフィックをプロキシ経由でルーティング | インターネットに直接アクセスできない企業ネットワーク |

**Git とコードの追跡**

| 設定 | 何をするか | いつ使うか |
|---------|-------------|----------------|
| `git_commit` / `git_remote` | 自動検出された Git の状態を上書き | 実行前に Git 情報を削除する CI 環境 |
| `disable_code` | コード取得を完全に無効化 | アップロードできない独自コード |
| `disable_git` | Git 状態の取得をスキップ | Git ではない環境、または Git 追跡のオーバーヘッドが問題になる場合 |

**分散学習**

| 設定 | 何をするか | いつ使うか |
|---------|-------------|----------------|
| `x_label` | shared モードでのこのノードのラベル | ログやシステムメトリクスで各ノードを区別（先ほど `"primary"`、`"worker_1"` を使いました） |
| `x_primary` | メインプロセスは `True`、worker は `False` | primary が設定・テレメトリのアップロードを担当。worker はメトリクスのみ送信 |
| `x_update_finish_state` | このプロセスが Run を完了状態にできるかを制御 | 早すぎる完了を防ぐため、worker では `False` に設定 |

**高度な設定**

| 設定 | 何をするか | いつ使うか |
|---------|-------------|----------------|
| `x_skip_transaction_log` | オンライン Run のローカルトランザクションログをスキップ | クラッシュリカバリを犠牲にディスク I/O を削減 |
| `x_stats_gpu_device_ids` | モニターする GPU インデックスのリスト（例: `[0, 1]`） | 一部の GPU だけを自分で所有している共有マシン |
| `reinit="create_new"` | 同一プロセス内で複数のアクティブな Run を許可 | 1 つのスクリプト内での並列実験（wandb>=0.19.10） |

## 16. まとめ

ここまでに作ったものを振り返り、次のステップを議論しましょう。

ここまでに取り上げた内容:

  1. セットアップ - 環境設定、W&B ログイン、config オブジェクト
  2. データと Artifacts - ローカルにプリロードされたデータ、use_artifact() でのリネージ
  3. データ探索 - EDA テーブル、画像統計、グルーピングとフィルタリング
  4. モデル学習 - Run の構造、tags、groups、commit=False、define_metric()、混合精度、TTL
  5. 視覚的ロギング - 予測画像、テーブル、ROC 曲線、クラス別メトリクス
  6. Run の再開 - ID で再開、シームレスに学習を継続
  7. オフラインモード - オフライン Run、同期
  8. モデル Artifact - モデル Artifact、Reference Artifact、エイリアス、TTL
  9. Registry - ベースラインをステージング、昇格前に確認
 10. Sweeps - 探索空間、ランダム探索、Sweep vs ベースライン、勝者を昇格
 11. Sweep の結果 - Sweep とベースラインの比較、勝者をプロダクションへ昇格
 12. Automations - CI/CD ループ、Registry トリガー、自動化ワークフロー
 13. プログラマティック API（オプション） - Public API クエリ、フィルタ、学習曲線
 14. プログラマティックレポート（オプション） - Reports API、ブロック、PanelGrid
 15. SDK 設定リファレンス（オプション） - ネットワーク、git、分散学習